# OncoEvidence — End-to-End Reproducible Workflow (Google Colab)

**Mechanism-Guided AI Evidence Triage for Cancer Drug Repurposing**

> **Core question.** For cancer drug repurposing, can a biomedical knowledge graph
> plus a citation-grounded LLM agent prioritize candidates by *both* predicted
> therapeutic relevance *and* mechanistic plausibility — and when does the graph
> add value over a strong tabular baseline?

This notebook runs the **entire** OncoEvidence pipeline from scratch, top to bottom:

1. **Environment setup** — clone the repo + install dependencies.
2. **Data collection** — download the PrimeKG knowledge graph (~980 MB) and build a
   PyTorch-Geometric `HeteroData` object + shared text node features.
3. **Aim 1 — Candidate generation / link benchmark** — GNN vs tuned XGBoost vs MLP
   vs KGE across 3 regimes, **plus topology & relation ablations** *(the honest
   negative result: a tuned tabular model matches the graph on link ranking)*.
4. **Decisive feature ablation** — does the GNN beat XGBoost once name/text
   semantics are removed (structural-only features)?
5. **Aim 2 — Mechanism extraction** — multi-hop drug→target→…→cancer MOA paths.
6. **Aim 4 — Mechanism separation** — do mechanism paths separate true indications
   from random pairs? (+ a hard-negative stress test).
7. **Aim 3 — LLM evidence verification** — citation-grounded LLM-as-judge over
   Europe PMC abstracts *(optional; needs an API key)*.
8. **Finding 4 — Blinded mechanism recovery** — the graph's one genuine edge over
   a tabular model.
9. **Deliverable** — a vetted oncology repurposing shortlist.

---

### How to run

- **Runtime → Change runtime type → GPU** is strongly recommended (T4 is enough).
- Run the cells top to bottom. The very next cell exposes two switches:
  - `FAST_MODE = True` runs a quick end-to-end smoke pass (1 seed, reduced epochs,
    hashing features) so the whole notebook finishes in a reasonable time. The
    **published numbers** require `FAST_MODE = False` (5 seeds, real
    SentenceTransformer features, XGBoost tuning) and take a few hours.
  - `RUN_LLM = True` enables the LLM verifier (needs `ONCO_LLM_API_KEY`).
- Each section also prints the **committed full-run results** so you can compare
  your run against the paper numbers even in fast mode.

> *All predictions are hypothesis-generating and not medical advice.*

## 0. Configuration switches

Edit these, then run the rest of the notebook top to bottom.

In [ ]:
# --- master switches -------------------------------------------------------
FAST_MODE = True   # True: quick smoke pass. False: full published reproduction (hours).
RUN_LLM   = False  # True: run the LLM evidence verifier (needs ONCO_LLM_API_KEY below).

# Optional: paste an OpenAI-compatible key (e.g. OpenRouter) to enable LLM dossiers.
# Leave blank to use the deterministic lexical-grounding fallback instead.
ONCO_LLM_API_KEY  = ""                                  # e.g. "sk-or-..."
ONCO_LLM_BASE_URL = "https://openrouter.ai/api/v1"      # default OpenAI: https://api.openai.com/v1
ONCO_LLM_MODEL    = "openai/gpt-4o-mini"

print(f"FAST_MODE={FAST_MODE} | RUN_LLM={RUN_LLM}")

## 1. Environment setup

Clone the project (on Colab) or locate it (if you already have it), then install
the Python dependencies. On Colab, PyTorch ships pre-installed; we only add the
extra libraries (PyG, XGBoost, Optuna, SentenceTransformers, …).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akshusuba/research.git"   # OncoEvidence lives in research/oncorepurpose

def find_project_root():
    "Walk up from CWD looking for the oncorepurpose package."
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "oncorepurpose" / "config.py").exists():
            return base
    return None

root = find_project_root()
if root is None:
    if not Path("research").exists():
        print("Cloning repo ...")
        # For a PRIVATE repo, use: REPO_URL = "https://<TOKEN>@github.com/akshusuba/research.git"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = (Path("research") / "oncorepurpose").resolve()

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)

# Use the kernel's own interpreter to run the scripts (robust on Colab & locally,
# where the executable may be `python3` rather than `python`).
PY = sys.executable

print("Project root:", root)
print("Python:", PY)
print("Contents:", sorted(p.name for p in root.iterdir()))

In [ ]:
# Install dependencies. On Colab torch is preinstalled, so we don't reinstall it
# (avoids breaking the CUDA build). PyG's pure-python layers used here work with
# any torch 2.x. If you are NOT on Colab and lack torch, also `pip install torch`.
%pip install -q torch-geometric xgboost optuna "sentence-transformers>=2.2.0" "transformers>=4.40,<5" scikit-learn pandas numpy scipy matplotlib seaborn requests tqdm pyyaml
print("Dependencies installed.")

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > GPU for speed.")

## 2. Data collection — PrimeKG

[PrimeKG](https://github.com/mims-harvard/PrimeKG) (Harvard) is a precision-medicine
knowledge graph. We download the raw `kg.csv` (~980 MB, from Harvard Dataverse),
parse it into a heterogeneous graph (`HeteroData`), and attach **shared**
SentenceTransformer node features that are fed to *both* the GNN and the XGBoost
baseline — so any performance gap is attributable to graph topology, not node
content.

> The download is large and only happens once; subsequent runs reuse the cached
> `data/kg.csv`, `data/primekg_hetero.pt`, and `models/*_features.pt`.

In [ ]:
# 2a. Download the PrimeKG raw edge list (skips if already present).
from oncorepurpose.data.download import download_primekg
download_primekg()

In [ ]:
# 2b. Parse kg.csv -> PyG HeteroData (cached to data/primekg_hetero.pt).
from oncorepurpose.config import HETERODATA_PT
from oncorepurpose.data.build_graph import build_hetero_from_primekg

if HETERODATA_PT.exists():
    print("HeteroData already built ->", HETERODATA_PT)
else:
    build_hetero_from_primekg()   # ~a few minutes on first run

In [ ]:
# 2c. Load the graph + attach shared node features, and print a summary.
# FAST_MODE uses the deterministic hashing features (instant); the full run uses
# real SentenceTransformer embeddings (cached after the first build).
from oncorepurpose.datasets import load_primekg, graph_summary

data, targets = load_primekg(with_features=True, force_fallback_features=FAST_MODE)
print(graph_summary(data, targets))

## 3. Aim 1 — Candidate generation: the link-prediction benchmark (+ ablations)

We rank `drug --indication--> cancer` pairs with four models across three regimes,
plus **topology ablations** (intact / shuffled / empty graph) and **relation
ablations** (dropping drug–protein, disease–protein, pathway edge families).

| Model | Regimes |
|---|---|
| HeteroGNN (ours), tuned XGBoost, FeatureMLP, DistMult-KGE | transductive · inductive cold-disease (oncology) · inductive cold-drug |

**Honest finding:** with a properly tuned XGBoost on the same text features, the
GNN no longer wins on link ranking — a rich name embedding lets the tabular model
take a semantic-similarity shortcut. This is the result that motivates the
mechanism-aware reframing in the later sections.

*Fast mode runs 1 seed with hashing features; the full run is 5 seeds with real
features + XGBoost tuning and writes `results/oncorepurpose.json`.*

In [ ]:
SMOKE = "--smoke" if FAST_MODE else ""
!{PY} scripts/run_experiment.py {SMOKE}

In [ ]:
# Show the results table + headline / ablation figures from THIS run.
from pathlib import Path
from IPython.display import Image, display

res_md = Path("results/oncorepurpose_smoke.md" if FAST_MODE else "results/oncorepurpose.md")
if res_md.exists():
    print(res_md.read_text())

for fig in ["figures/main_results.png", "figures/ablation_topology.png"]:
    if Path(fig).exists():
        display(Image(fig))

In [ ]:
# For reference: the COMMITTED full 5-seed published table (Finding 1).
from pathlib import Path
p = Path("results/oncorepurpose.md")
print("=== Published full-run benchmark (committed) ===\n")
print(p.read_text() if p.exists() else "(committed results not present in this checkout)")

## 4. Decisive feature ablation — does the GNN ever beat XGBoost?

We re-run the cold-disease (oncology) regime under three **shared** feature
settings: text (semantic), structural (graph-degree only, no semantics), and
random (reference only). The question: when XGBoost can no longer exploit
name-embedding similarity, does graph topology let the GNN pull ahead?

*Writes `results/feature_ablation.json` (or `..._smoke.json`).*

In [ ]:
!{PY} scripts/feature_ablation.py {SMOKE}

In [ ]:
import json
from pathlib import Path
fa = Path("results/feature_ablation_smoke.json" if FAST_MODE else "results/feature_ablation.json")
if fa.exists():
    d = json.loads(fa.read_text())
    print("AUROC grid (mean over seeds):")
    print(json.dumps(d["auroc_grid_mean"], indent=2))
    print("\n" + d["interpretation"])

## 5. Aim 2 — Mechanism extraction

XGBoost can score a pair but cannot produce a traceable mechanism. The graph can.
For true oncology indications, the multi-hop extractor recovers textbook
direct-target mechanisms (e.g. `Quizartinib → FLT3 ← myeloid leukemia`), while
random drug–cancer pairs mostly yield no mechanistic path. Promiscuous carrier
hubs (e.g. albumin) are filtered out.

In [ ]:
!{PY} scripts/mechanism_demo.py

## 6. Aim 4 — Mechanism structure separates true vs random pairs

The falsifiable claim, tested **LLM-free** on the graph signal over 400 true
oncology indications vs 400 random drug–cancer pairs: mechanism paths separate the
two groups (published **AUROC 0.879**). This cell also runs a small
literature-grounding sample and a best-effort **DrugMechDB** agreement check
(both need network access; they degrade gracefully if offline).

*Writes `results/mechanism_eval.json` + `figures/mechanism_eval.png`.*

In [ ]:
!{PY} scripts/evaluate_mechanism.py

In [ ]:
from pathlib import Path
from IPython.display import Image, display
mm = Path("results/mechanism_eval.md")
if mm.exists():
    print(mm.read_text())
if Path("figures/mechanism_eval.png").exists():
    display(Image("figures/mechanism_eval.png"))

### 6b. Hard-negative stress test

Reviewers object that random negatives are too easy. We recompute the separation
AUROC against progressively harder, fairer negatives (degree-matched, oncology
drugs vs cancers they don't treat, shared-target), plus mechanism-score ablations.

*Writes `results/hard_negatives_eval.json`.*

In [ ]:
!{PY} scripts/evaluate_hard_negatives.py

## 7. Aim 3 — LLM evidence verification (optional)

The verifier retrieves Europe PMC **abstracts** and grades each mechanism path as
**supported / weak / contradicted / unknown** with quoted evidence — via an LLM
when `ONCO_LLM_API_KEY` is set, and a lexical-grounding fallback otherwise.

This cell only runs if `RUN_LLM = True` **and** a key is provided in section 0.
Published run (`gpt-4o-mini`, 50 true vs 50 random): LLM marks 11 *supported* among
true pairs and only 1 among random; DrugMechDB precision of *supported* is 0.857
(LLM) vs 0.591 (lexical).

In [ ]:
import os
if RUN_LLM and ONCO_LLM_API_KEY:
    os.environ["ONCO_LLM_API_KEY"]  = ONCO_LLM_API_KEY
    os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
    os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL
    !{PY} scripts/verify_llm_eval.py
else:
    print("Skipping live LLM verification (set RUN_LLM=True and paste a key in section 0).")
    # Show the committed published LLM verification summary instead:
    import json
    from pathlib import Path
    p = Path("results/verify_llm_eval.json")
    if p.exists():
        d = json.loads(p.read_text())
        print("\n=== Committed published LLM verification summary ===")
        print("model:", d.get("model"))
        print("LLM grade dist:", d.get("llm_dist"))
        print("supported rate:", d.get("supported_rate"))
        print("DrugMechDB precision of 'supported':", d.get("dmdb_precision_supported"))

## 8. Finding 4 — Blinded mechanism recovery (the graph's genuine edge)

We train the GNN jointly (link BCE + a contrastive loss ranking the curated
DrugMechDB bridge gene above degree-matched decoys) and ask it to *name the bridge
gene* for held-out cold-disease pairs. With the direct drug→target edge removed
(mechanism-blinded), trivial lookup and degree priors collapse to ~0 while the
joint GNN still recovers the bridge gene — an axis a tabular model cannot touch.

*Needs network access for DrugMechDB. Writes `results/mechanism_recovery_eval.json`.*

In [ ]:
!{PY} scripts/evaluate_mechanism_recovery.py {SMOKE}

In [ ]:
import json
from pathlib import Path
p = Path("results/mechanism_recovery_eval.json")
if p.exists():
    d = json.loads(p.read_text())
    print(d.get("interpretation", "(no interpretation field)"))

## 9. Deliverable — vetted oncology repurposing shortlist

Train a deployment GNN on indication edges, rank novel drug candidates for
selected cancers, extract multi-hop KG rationales, retrieve supporting literature,
and (optionally) add an LLM evidence dossier — writing a ranked markdown + JSON
shortlist.

*Fast mode uses a short GNN training run with hashing features and no LLM.*

In [ ]:
if FAST_MODE:
    REPORT_ARGS = '--diseases glioblastoma "pancreatic cancer" --top-k 3 --gnn-epochs 5 --fallback-features --no-llm'
else:
    REPORT_ARGS = '--diseases glioblastoma "pancreatic cancer" --top-k 5'
    if RUN_LLM and ONCO_LLM_API_KEY:
        import os
        os.environ["ONCO_LLM_API_KEY"]  = ONCO_LLM_API_KEY
        os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
        os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL
    else:
        REPORT_ARGS += " --no-llm"

!{PY} scripts/generate_report.py {REPORT_ARGS}

In [ ]:
from pathlib import Path
p = Path("results/repurposing_shortlist.md")
if p.exists():
    txt = p.read_text()
    print(txt[:4000] + ("\n... (truncated; see results/repurposing_shortlist.md)" if len(txt) > 4000 else ""))

## Wrap-up

You've run the full OncoEvidence workflow end to end. Artifacts written to disk:

- `data/` — PrimeKG `kg.csv`, parsed `primekg_hetero.pt`
- `models/` — cached node features + LLM response cache
- `results/` — every `*.json` / `*.md` results file produced above
- `figures/` — `main_results.png`, `ablation_topology.png`, `mechanism_eval.png`

**To reproduce the published numbers**, set `FAST_MODE = False` (5 seeds, real
SentenceTransformer features, XGBoost tuning) and, if you have a key, `RUN_LLM = True`.
The full run takes a few hours and is GPU-bound.

**Summary of findings**
1. On link ranking alone, a tuned XGBoost matches/beats the GNN — the graph isn't
   necessary for that task.
2. But the graph carries traceable mechanism a tabular model cannot produce.
3. Mechanism structure separates true vs random oncology pairs (AUROC 0.879).
4. The graph's one genuine edge: blinded mechanism recovery (R@10 0.25 where
   trivial/link-only baselines hit 0).

> *All predictions are hypothesis-generating and not medical advice.*